In [1]:
FAST_MODE = True

In [ ]:
!pip install captum

In [ ]:
!pip install numpy datasets

In [ ]:
!pip install transformers[torch] accelerate -U

In [ ]:
!pip install datasets evaluate

In [2]:
import transformers, accelerate
print(transformers.__version__)
print(accelerate.__version__)

5.3.0
1.13.0


In [3]:
pip show torch

Name: torch
Version: 2.11.0+cu126
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License: BSD-3-Clause
Location: c:\users\ps3\documents\sem2\nlp\capstone\nlp-proj\lib\site-packages
Requires: filelock, fsspec, jinja2, networkx, setuptools, sympy, typing-extensions
Required-by: accelerate, captum, torchvision
Note: you may need to restart the kernel to use updated packages.


In [4]:

import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# from captum.attr import IntegratedGradients

In [5]:
import torch
print(torch.__version__)
# print(torchvision.__version__)

2.11.0+cu126


In [ ]:
from transformers.trainer import Trainer

In [6]:
import pandas as pd
import os

df = pd.read_csv(f"C:/Users/PS3/Documents/NLP_Proj/fake-job-detection-nlp-bilstm/fraud posting/fake_job_postings_augmented.csv")

print(f"Dataset loaded: {df.shape}")
print(f"Fraud rate: {df['fraudulent'].mean()*100:.2f}%")
print(df['fraudulent'].value_counts())

Dataset loaded: (29880, 18)
Fraud rate: 43.06%
fraudulent
0    17014
1    12866
Name: count, dtype: int64


In [7]:
def combine_text_fields(row):
    fields = [
        'title', 'location', 'company_profile', 'description',
        'requirements', 'benefits', 'required_experience',
        'required_education', 'industry', 'function'
    ]

    text_parts = []
    for field in fields:
        if pd.notna(row[field]):
            text_parts.append(str(row[field]))

    return ' '.join(text_parts) if text_parts else "unknown job"

print("Combining text fields...")
df['combined_text'] = df.apply(combine_text_fields, axis=1)
print("Done")

Combining text fields...
Done


In [8]:
df.head()

,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent,combined_text
0,5941.0,Full Stack Engineer,"US, CA, Palo Alto",NaN,NaN,Declara is focused on bringing data to life. O...,"Declara is looking for a super-sharp, ambitiou...",Strong (5+ years) experience in a dynamic lang...,NaN,0,1,1,NaN,NaN,NaN,NaN,NaN,0,"Full Stack Engineer US, CA, Palo Alto Declara ..."
1,NaN,Business Development Representative,"US, CO, Denver",Logistics,3000-6000 per month,Omega Marketing Associates is a full-service m...,As a Business Development Representative you w...,Excellent organisational and communication ski...,100% remote. Flexible schedule. Health coverag...,1,0,0,Temporary,Associate,Some College Coursework Completed,Staffing and Recruiting,Finance,1,"Business Development Representative US, CO, De..."
2,12972.0,Senior Business Development Manager Europe,"DE, BY, Munich",NaN,NaN,hello worldtalents23_ drives the change in dig...,At our company we believe that unnecessarily w...,Ideally you already have many years of operati...,If you are passionate about the dynamic start-...,0,1,0,Full-time,NaN,NaN,NaN,NaN,0,"Senior Business Development Manager Europe DE,..."
3,NaN,Payment Coordinator – Remote,"US, MA, Boston",Administration,60000-85000,Quantum Services Inc is a global payment proce...,This role involves receiving international cli...,Personal or business bank account required. Ab...,Competitive processing fees. Fully remote work...,0,0,0,Temporary,Director,High School or equivalent,Staffing and Recruiting,Unknown,1,"Payment Coordinator – Remote US, MA, Boston Qu..."
4,15673.0,Junior Data Analyst,"GB, RIC, Twickenham",NaN,20000-24000,With an exceptional record of over 50% growth ...,"WorldStores, the UK’s leading online retailer ...",Requirements:Idea on Google AdWords and PPC/PL...,NaN,0,1,0,Full-time,Entry level,Bachelor's Degree,Retail,Data Analyst,0,"Junior Data Analyst GB, RIC, Twickenham With ..."


In [ ]:
df = df[['combined_text', 'fraudulent']].dropna()
df['combined_text'] = df['combined_text'].astype(str)

print(f"After dropna: {df.shape}")
print(f"Fraud rate: {df['fraudulent'].mean()*100:.2f}%")

After dropna: (29880, 2)
Fraud rate: 43.06%


In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['fraudulent']
)

# Split temp 50/50 → 10% val, 10% test
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df['fraudulent']
)

print(f"Train: {len(train_df)} ({len(train_df)/len(df)*100:.0f}%)")
print(f"Val:   {len(val_df)} ({len(val_df)/len(df)*100:.0f}%)")
print(f"Test:  {len(test_df)} ({len(test_df)/len(df)*100:.0f}%)")
print(f"\nFraud rates → Train: {train_df['fraudulent'].mean()*100:.1f}%  Val: {val_df['fraudulent'].mean()*100:.1f}%  Test: {test_df['fraudulent'].mean()*100:.1f}%")

Train: 23904 (80%)
Val:   2988 (10%)
Test:  2988 (10%)

Fraud rates → Train: 43.1%  Val: 43.1%  Test: 43.0%


In [11]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/MiniLM-L12-H384-uncased")

model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/MiniLM-L12-H384-uncased",
    num_labels=2,
    id2label={0: "legitimate", 1: "fraud"},
    label2id={"legitimate": 0, "fraud": 1}
)

print("Model loaded with correct label mapping:")
print(f"  Class 0: {model.config.id2label[0]}")
print(f"  Class 1: {model.config.id2label[1]}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice: {device}")
model = model.to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: microsoft/MiniLM-L12-H384-uncased
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded with correct label mapping:
  Class 0: legitimate
  Class 1: fraud

Device: cuda


In [12]:
def tokenize(batch):
    return tokenizer(
        batch["combined_text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

train_ds = Dataset.from_pandas(train_df).map(tokenize, batched=True)
val_ds   = Dataset.from_pandas(val_df).map(tokenize, batched=True)
test_ds  = Dataset.from_pandas(test_df).map(tokenize, batched=True)

# Rename label column
train_ds = train_ds.rename_column("fraudulent", "labels")
val_ds   = val_ds.rename_column("fraudulent", "labels")
test_ds  = test_ds.rename_column("fraudulent", "labels")

# Set torch format
for ds in [train_ds, val_ds, test_ds]:
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print(f"Train: {len(train_ds)} samples")
print(f"Val:   {len(val_ds)} samples")
print(f"Test:  {len(test_ds)} samples")

Map:   0%|          | 0/23904 [00:00<?, ? examples/s]

Map:   0%|          | 0/2988 [00:00<?, ? examples/s]

Map:   0%|          | 0/2988 [00:00<?, ? examples/s]

Train: 23904 samples
Val:   2988 samples
Test:  2988 samples


In [ ]:
!pip install accelerate

In [ ]:
import sys
print("Python executable:", sys.executable)

import transformers
print("transformers version:", transformers.__version__)

try:
    import accelerate
    print("accelerate version:", accelerate.__version__)
    print("accelerate file:", accelerate.__file__)
except Exception as e:
    print("accelerate import failed:", repr(e))

In [ ]:
import sys
print("Python:", sys.executable)

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

import transformers
print("transformers:", transformers.__version__)

import accelerate
print("accelerate:", accelerate.__version__)

from transformers.utils import is_accelerate_available, is_torch_available
print("is_torch_available():", is_torch_available())
print("is_accelerate_available():", is_accelerate_available())

In [16]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1 if FAST_MODE else 3,
    per_device_train_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),   # fp16 only if GPU available
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)

print("Starting training...")
trainer.train()
print("\nTraining complete!")

Starting training...


Epoch,Training Loss,Validation Loss
1,0.079759,0.065658


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Training complete!


In [ ]:
from sklearn.metrics import f1_score, classification_report, confusion_matrix

model.eval()
fraud_probs = []
true_labels = []

with torch.no_grad():
    for batch in test_ds:
        inputs = {
            'input_ids':      batch['input_ids'].unsqueeze(0).to(device),
            'attention_mask': batch['attention_mask'].unsqueeze(0).to(device)
        }
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        fraud_probs.append(probs[0][1].item())
        true_labels.append(batch['labels'].item())

best_f1 = 0
best_threshold = 0.5

for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    preds = [1 if p > threshold else 0 for p in fraud_probs]
    f1 = f1_score(true_labels, preds, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

print(f"Best threshold: {best_threshold}")
predictions = [1 if p > best_threshold else 0 for p in fraud_probs]

print("\nClassification Report:")
print(classification_report(true_labels, predictions, target_names=['Legitimate', 'Fraud'], digits=3))

cm = confusion_matrix(true_labels, predictions)
tn, fp, fn, tp = cm.ravel()

print(f"Confusion Matrix:")
print(f"                Predicted")
print(f"                Legit  Fraud")
print(f"Actual Legit    {tn:4d}  {fp:4d}")
print(f"       Fraud    {fn:4d}  {tp:4d}")
print(f"\nFraud Recall:    {tp/(tp+fn)*100:.1f}%")
print(f"Precision:       {tp/(tp+fp)*100:.1f}%")
print(f"F1-Score:        {best_f1:.3f}")

Best threshold: 0.6

Classification Report:
              precision    recall  f1-score   support

  Legitimate      0.970     0.998     0.983      1702
       Fraud      0.997     0.959     0.977      1286

    accuracy                          0.981      2988
   macro avg      0.983     0.978     0.980      2988
weighted avg      0.981     0.981     0.981      2988

Confusion Matrix:
                Predicted
                Legit  Fraud
Actual Legit    1698     4
       Fraud      53  1233

Fraud Recall:    95.9%
Precision:       99.7%
F1-Score:        0.977


In [18]:
from pathlib import Path

save_dir = Path('../models/model_miniLM_final')
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"Model and tokenizer saved to {save_dir}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved to ..\models\model_miniLM_final


In [19]:
!pip install captum

In [20]:
from captum.attr import IntegratedGradients

In [ ]:
def forward_pass(input_ids, attention_mask):
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    return F.softmax(outputs.logits, dim=1)[:, 1]

def forward_pass_embeds(inputs_embeds, attention_mask):
    outputs = model(inputs_embeds=inputs_embeds, attention_mask=attention_mask)
    return F.softmax(outputs.logits, dim=1)[:, 1]

ig = IntegratedGradients(forward_pass_embeds)

word_embeddings = model.base_model.embeddings.word_embeddings

def get_attributions(text, n_steps=50):
    encoded = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=256,
        padding='max_length'
    ).to(device)

    inputs_embeds  = word_embeddings(encoded['input_ids']).detach().requires_grad_(True)
    baseline_embeds = torch.zeros_like(inputs_embeds).to(device)

    attributions, delta = ig.attribute(
        inputs=inputs_embeds,
        baselines=baseline_embeds,
        additional_forward_args=(encoded['attention_mask'],),
        return_convergence_delta=True,
        n_steps=n_steps
    )

    attributions_sum  = attributions.sum(dim=-1).squeeze(0)
    attributions_norm = attributions_sum / torch.norm(attributions_sum)
    tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])

    with torch.no_grad():
        fraud_prob = forward_pass(encoded['input_ids'], encoded['attention_mask']).item()

    return tokens, attributions_norm.detach().cpu().numpy(), fraud_prob, delta.item()


def visualize_attributions(text, attributions, tokens, top_k=15):
    filtered = [(tok, attr) for tok, attr in zip(tokens, attributions)
                if tok not in ['[CLS]', '[SEP]', '[PAD]']]
    sorted_attr = sorted(filtered, key=lambda x: x[1], reverse=True)

    print("\n" + "="*80)
    print("WORD ATTRIBUTION ANALYSIS")
    print("="*80)
    print(f"\nText: {text[:150]}...")
    print(f"\nTop {top_k} Fraud Indicators (positive attribution):")
    for i, (token, attr) in enumerate(sorted_attr[:top_k], 1):
        print(f"  {i:2d}. {token:20s} -> {attr:+.4f}")
    print(f"\nTop 5 Legitimacy Indicators (negative attribution):")
    for i, (token, attr) in enumerate(sorted_attr[-5:], 1):
        print(f"  {i:2d}. {token:20s} -> {attr:+.4f}")

print("IG setup complete.")

IG setup complete.


In [22]:
import torch.nn.functional as F

print("="*80)
print("EXAMPLE 1: HIGH-CONFIDENCE FRAUD DETECTION")
print("="*80)

fraud_examples = test_df[test_df['fraudulent'] == 1]
text = fraud_examples.iloc[0]['combined_text']

tokens, attributions, fraud_prob, delta = get_attributions(text)

print(f"\nModel Prediction: {fraud_prob:.1%} fraud probability")
print(f"Actual Label: FRAUD")
print(f"{'Correct!' if fraud_prob > best_threshold else 'Missed!'}")

visualize_attributions(text, attributions, tokens, top_k=15)
print(f"\nConvergence Delta: {delta:.6f} (closer to 0 is better)")

EXAMPLE 1: HIGH-CONFIDENCE FRAUD DETECTION

Model Prediction: 99.1% fraud probability
Actual Label: FRAUD
Correct!

WORD ATTRIBUTION ANALYSIS

Text: Home Logistics Coordinator GH, AA, Accra Rapid Enterprises connects international shoppers with domestic shipping agents, ensuring secure and timely d...

Top 15 Fraud Indicators (positive attribution):
   1. .                    -> +0.2396
   2. .                    -> +0.2327
   3. .                    -> +0.2164
   4. .                    -> +0.2109
   5. .                    -> +0.1865
   6. neighbours           -> +0.1645
   7. .                    -> +0.1640
   8. disclose             -> +0.1630
   9. parcel               -> +0.1606
  10. .                    -> +0.1588
  11. nearest              -> +0.1568
  12. .                    -> +0.1506
  13. contents             -> +0.1505
  14. ,                    -> +0.1453
  15. .                    -> +0.1315

Top 5 Legitimacy Indicators (negative attribution):
   1. to                 

In [23]:
print("="*80)
print("EXAMPLE 2: HIGH-CONFIDENCE LEGITIMATE DETECTION")
print("="*80)

legit_examples = test_df[test_df['fraudulent'] == 0]
text_legit = legit_examples.iloc[0]['combined_text']

tokens_l, attributions_l, fraud_prob_l, delta_l = get_attributions(text_legit)

print(f"\nModel Prediction: {fraud_prob_l:.1%} fraud probability")
print(f"Actual Label: LEGITIMATE")
print(f"{'Correct!' if fraud_prob_l < best_threshold else 'False positive!'}")

visualize_attributions(text_legit, attributions_l, tokens_l, top_k=15)
print(f"\nConvergence Delta: {delta_l:.6f}")

EXAMPLE 2: HIGH-CONFIDENCE LEGITIMATE DETECTION

Model Prediction: 1.2% fraud probability
Actual Label: LEGITIMATE
Correct!

WORD ATTRIBUTION ANALYSIS

Text: Senior SQL & Database Developer GB, LND, Shoreditch Adthena is the UK’s leading competitive intelligence service for Google search advertisers. Adthen...

Top 15 Fraud Indicators (positive attribution):
   1. ?                    -> +0.0822
   2. spend                -> +0.0726
   3. for                  -> +0.0714
   4. "                    -> +0.0673
   5. .                    -> +0.0640
   6. -                    -> +0.0582
   7. want                 -> +0.0571
   8. search               -> +0.0565
   9. ,                    -> +0.0555
  10. _                    -> +0.0543
  11. job                  -> +0.0541
  12. join                 -> +0.0528
  13. another              -> +0.0501
  14. #                    -> +0.0475
  15. service              -> +0.0473

Top 5 Legitimacy Indicators (negative attribution):
   1. work      

In [24]:
print("="*80)
print("EXAMPLE 3: FALSE NEGATIVE (MISSED FRAUD)")
print("="*80)

missed_frauds_idx = [
    i for i, (true, pred) in enumerate(zip(true_labels, predictions))
    if true == 1 and pred == 0
]

if missed_frauds_idx:
    fn_example = test_df.iloc[missed_frauds_idx[0]]
    text_fn = fn_example['combined_text']

    tokens_fn, attributions_fn, fraud_prob_fn, delta_fn = get_attributions(text_fn)

    print(f"\nModel Prediction: {fraud_prob_fn:.1%} fraud probability (below {best_threshold} threshold)")
    print(f"Actual Label: FRAUD")
    print(f"MISSED - Why did the model fail?")

    visualize_attributions(text_fn, attributions_fn, tokens_fn, top_k=15)
    print(f"\nConvergence Delta: {delta_fn:.6f}")
    print("\nAnalysis: This fraud likely lacks obvious fraud keywords and appears more professional.")
else:
    print("\nNo false negatives found in test set!")

EXAMPLE 3: FALSE NEGATIVE (MISSED FRAUD)

Model Prediction: 2.1% fraud probability (below 0.6 threshold)
Actual Label: FRAUD
MISSED - Why did the model fail?

WORD ATTRIBUTION ANALYSIS

Text: QC Inspector US, TX, Houston Aker Solutions is a global provider of products, systems and services to the oil and gas industry. Our engineering, desig...

Top 15 Fraud Indicators (positive attribution):
   1. .                    -> +0.0904
   2. not                  -> +0.0841
   3. only                 -> +0.0774
   4. .                    -> +0.0707
   5. within               -> +0.0657
   6. position             -> +0.0635
   7. a                    -> +0.0571
   8. ,                    -> +0.0569
   9. position             -> +0.0521
  10. go                   -> +0.0507
  11. ,                    -> +0.0496
  12. ,                    -> +0.0477
  13. to                   -> +0.0476
  14. to                   -> +0.0458
  15. looking              -> +0.0457

Top 5 Legitimacy Indicators (negat

In [ ]:
import numpy as np
print("="*80)
print("AGGREGATE ANALYSIS: TOP FRAUD INDICATORS ACROSS TEST SET")
print("="*80)

word_attributions = {}
fraud_samples = test_df[test_df['fraudulent'] == 1].head(50)
print(f"\nAnalyzing {len(fraud_samples)} fraud examples...")

for idx, row in fraud_samples.iterrows():
    tokens_a, attrs_a, _, _ = get_attributions(row['combined_text'], n_steps=25)

    for token, attr in zip(tokens_a, attrs_a):
        if token not in ['[CLS]', '[SEP]', '[PAD]'] and not token.startswith('##'):
            word_attributions.setdefault(token, []).append(attr)

mean_attributions = {
    word: np.mean(attrs)
    for word, attrs in word_attributions.items()
    if len(attrs) >= 3
}

sorted_words = sorted(mean_attributions.items(), key=lambda x: x[1], reverse=True)

print("\nTOP 20 FRAUD INDICATOR WORDS (across all fraud examples):")
for i, (word, attr) in enumerate(sorted_words[:20], 1):
    count = len(word_attributions[word])
    print(f"  {i:2d}. {word:20s} -> {attr:+.4f} (appeared in {count} examples)")

print("\nIntegrated Gradients analysis complete!")

AGGREGATE ANALYSIS: TOP FRAUD INDICATORS ACROSS TEST SET

Analyzing 50 fraud examples...

TOP 20 FRAUD INDICATOR WORDS (across all fraud examples):
   1. credit               -> +0.1762 (appeared in 3 examples)
   2. .                    -> +0.1687 (appeared in 696 examples)
   3. email                -> +0.1676 (appeared in 7 examples)
   4. retain               -> +0.1243 (appeared in 4 examples)
   5. details              -> +0.1233 (appeared in 4 examples)
   6. conducts             -> +0.1218 (appeared in 3 examples)
   7. seeking              -> +0.1196 (appeared in 4 examples)
   8. access               -> +0.1189 (appeared in 42 examples)
   9. #                    -> +0.1182 (appeared in 9 examples)
  10. prior                -> +0.1170 (appeared in 6 examples)
  11. involves             -> +0.1165 (appeared in 14 examples)
  12. included             -> +0.1116 (appeared in 5 examples)
  13. managing             -> +0.1115 (appeared in 4 examples)
  14. needed               ->

: 